In [0]:
# ================== Third take home assignment GR5069 Spring 2026 ========

# ================== Global Definitions ====================================

# # Packages to import
import pandas as pd
import mlflow
from sklearn.linear_model import LogisticRegression 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import os 
import matplotlib.pyplot as plt
import seaborn as sns
import tempfile
from sklearn.metrics import ConfusionMatrixDisplay
import tempfile

# Data Paths 
pitstops_path = "/Volumes/gr5069/raw/f1_data/pit_stops.csv"
results_path = "/Volumes/gr5069/raw/f1_data/results.csv"

In [0]:
# Load, join and format data

# Load pitstop data 
df_pitstops = pd.read_csv(pitstops_path)
df_pitstops["raceId"] = df_pitstops["raceId"].astype(int)
df_pitstops["driverId"] = df_pitstops["driverId"].astype(int)
df_pitstops["lap"] = df_pitstops["lap"].astype(int)
df_pitstops["stop"] = df_pitstops["stop"].astype(int)
df_pitstops.head()

# Load results data 
df_results = pd.read_csv(results_path)
df_results["raceId"] = df_results["raceId"].astype(int)
df_results["driverId"] = df_results["driverId"].astype(int)

# Join results and pitstop data on raceId and driverId
df_pitstops = df_pitstops.merge(df_results[["raceId", "driverId", "position", "grid"]], on=["raceId", "driverId"], how="left")

# Replace null values (those who did not finish the race) with NaN 
df_pitstops["position"] = df_pitstops["position"].replace("\\N", pd.NA)
df_pitstops["position"] = pd.to_numeric(df_pitstops["position"], errors="coerce")

# Replace null values in grid with NaN
df_pitstops["grid"] = df_pitstops["grid"].replace("\\N", pd.NA)
df_pitstops["grid"] = pd.to_numeric(df_pitstops["grid"], errors="coerce")

In [0]:
# Calculate feature and target variables for model training

# Aggregate features and target into one dataset
df = df_pitstops.groupby(["raceId", "driverId"]).agg(
    avg_duration=("milliseconds", "mean"), # average pitstop duration
    num_stops=("stop", "max"), # number of pitstops
    first_pit_lap=("lap", "min"), # first pitstop lap
    position=("position", "first"), # final position
    grid=("grid", "first") # starting grid position
).reset_index() 

# Create binary target variable — 1 if podium (top 3), 0 otherwise
df["podium"] = (df["position"] <= 3).astype(int)

# Drop rows with missing values (This model will not include drivers who did not finish the race)
df = df.dropna()

# Drop position column because it's no longer needed 
df = df.drop(columns=["position"])

# Display model dataset 
display(df.head(5))

In [0]:
# Separate feature and target variables into trainings and testing data 

# Define feature and target variables (X and y)
y = df["podium"]
X = df.drop(columns=["podium", "raceId", "driverId"])

# Train/test/split (defaults to .75/.25 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 123)

In [0]:
# Basic logistic regression model (with no defined parameters)
with mlflow.start_run(run_name="Baseline Model") as run:

    # Create model, train and predict
    log_model = LogisticRegression()
    log_model.fit(X_train, y_train)
    predicted_vals = log_model.predict(X_test)

    # Log model 
    mlflow.sklearn.log_model(log_model, "Baseline Model")

    # Create Model metrics 
    f1 = f1_score(y_test, predicted_vals)
    accuracy = accuracy_score(y_test, predicted_vals)
    precision = precision_score(y_test, predicted_vals)
    recall = recall_score(y_test, predicted_vals)

    # Log metrics 
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)

    runID = run.info.run_id
    experimentID = run.info.experiment_id
  
print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))

In [0]:
#Define function that logs parameters, metrics, and feature importance when running your model. 
 
def log_logistic(experimentID, run_name, params, X_train, X_test, y_train, y_test):

    with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:

        log_model = make_pipeline(StandardScaler(), LogisticRegression(**params))
        log_model.fit(X_train, y_train)
        predicted_vals = log_model.predict(X_test)

        mlflow.sklearn.log_model(log_model, "Basic logistic regression model")
        runID = run.info.run_id
        experimentID = run.info.experiment_id

        [mlflow.log_param(param, value) for param, value in params.items()]

        # Calculate metrics 
        f1 = f1_score(y_test, predicted_vals)
        accuracy = accuracy_score(y_test, predicted_vals)
        precision = precision_score(y_test, predicted_vals)
        recall = recall_score(y_test, predicted_vals)

        # Log metrics 
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)

        # Create feature importance 
        importance = pd.DataFrame(list(zip(X_train.columns, [0]*len(X_train.columns))),
                                  columns=["Feature", "Importance"]
                                 ).sort_values("Importance", ascending=False)

        # Log importances using a temporary file
        temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv", delete=False)
        temp_name = temp.name
        try:
            importance.to_csv(temp_name, index=False)
            mlflow.log_artifact(temp_name, "feature-importance.csv")
        finally:
            temp.close()

        # Create plot
        fig, ax = plt.subplots()

        ConfusionMatrixDisplay.from_predictions(y_test, predicted_vals, ax=ax)
        plt.title("Confusion Matrix — Podium Prediction")

        # Log plot using a temporary file 
        temp = tempfile.NamedTemporaryFile(prefix="confusion-matrix-", suffix=".png")
        temp_name = temp.name
        try:
            fig.savefig(temp_name)
            mlflow.log_artifact(temp_name, "confusion-matrix.png")
        finally:
            temp.close()

        display(fig)
    return run.info.run_id

In [0]:
# 1st run hyper parameters
params = {
    "C": 100,
    "penalty": "l2",
    "solver": "lbfgs",
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42
}

log_logistic(experimentID, "1st Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 2nd run 

params = {
    "C": 10,
    "penalty": "l2",
    "solver": "lbfgs",
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42
}

log_logistic(experimentID, "2nd Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 3rd run 
params = {
    "C": 1,
    "penalty": "l2",
    "solver": "lbfgs",
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42
}

log_logistic(experimentID, "3rd Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 4th run 
params = {
    "C": 0.1,
    "penalty": "l2",
    "solver": "lbfgs",
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42
}

log_logistic(experimentID, "4th Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 5th run 
params = {
    "C": 0.001,
    "penalty": "l2",
    "solver": "lbfgs",
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42
}
log_logistic(experimentID, "5th Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 6th run 

params = {
    "C": 100,
    "penalty": "l1",
    "solver": "liblinear",
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42
}

log_logistic(experimentID, "6th Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 7th run 

params = {
    "C": 10,
    "penalty": "l1",
    "solver": "liblinear",
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42
}

log_logistic(experimentID, "7th Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 8th run 

params = {
    "C": 1.0,
    "penalty": "l1",
    "solver": "liblinear",
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42
}

log_logistic(experimentID, "8th Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 9th run 

params = {
    "C": .01,
    "penalty": "l1",
    "solver": "liblinear",
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42
}

log_logistic(experimentID, "9th Run", params, X_train, X_test, y_train, y_test)

In [0]:
# Update parameters 10th run 

params = {
    "C": .001,
    "penalty": "l1",
    "solver": "liblinear",
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42
}

log_logistic(experimentID, "10th Run", params, X_train, X_test, y_train, y_test)